# 2 Baseline: Model Without Context

Purpose:
Establish a "before" snapshot so improvements later are visible, attributable, and defensible.
What we're doing

We will ask a small set of domain-specific questions without providing any of the customer documents. This simulates what happens when a customer "just tries" a model before connecting their data.

Why we're doing it

* To create a reference point
* To reveal the gap between fluency and groundedness
* To produce a baseline artifact we can later compare against RAG results

What to look for

* Confident language + incorrect facts
* Vague responses where specificity is required
* Inconsistent terminology across runs

Output

A small baseline dataset:
question → model answer → timestamp → metadata
 saved for later comparison and evaluation.

## 2.1 Setup a Key on MaaS

Before we can interact with any language model, we need an API key from the **Model as a Service (MaaS)** platform. This key authenticates every request we send and tracks usage against your account.

**Steps:**

1. Open the Red Hat Demo Platform MaaS service home page in your browser:
   https://litellm-prod-frontend.apps.maas.redhatworkshops.io/home
2. Log in using your **Red Hat SSO** credentials.
3. Navigate to **API Keys** and click **Create Key**.
4. Give your key a descriptive name (e.g., `q1-workshop`) and click **Generate**.
5. **Copy the key immediately** — it will only be displayed once.

> **Tip:** Keep this browser tab open. You will need both the API key and the endpoint URL in the next step.

## 2.1.1 Create your .env file

With your API key in hand, we need to store it — along with the endpoint URL — in a `.env` file at the root of the project directory. This file keeps sensitive configuration out of our notebook code and makes it easy to swap credentials later without modifying any cells.

The `.env` file will contain two values:

| Variable | Description |
|---|---|
| `API_KEY` | The key you just generated on the MaaS platform |
| `ENDPOINT_BASE` | The base URL for the MaaS API (found on the key details screen) |

In the cell below, **uncomment the lines** and replace `APIKEY` with the key you copied and update the endpoint URL if it differs from the default. Then run the cell to write the file.

In [2]:
# !echo 'API_KEY=APIKEY' > ../.env
# !echo 'ENDPOINT_BASE=https://litellm-prod.apps.maas.redhatworkshops.io/v1' >> ../.env

Check the that the data is correct

## 2.2 Testing the connection and API Key


### Load variables from the .env file

In [4]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base

print(f"Endpoint: {endpoint_base}")
print(f"API Key:  {key[:8]}...")


Endpoint: https://litellm-prod.apps.maas.redhatworkshops.io/v1
API Key:  sk-UFHcL...


In [5]:
import requests
url_models = f"{endpoint_base}/models"
url_chat = f"{endpoint_base}/chat/completions"

## Test the Connection to MaaS Server`

The following code will test the connection to the MaaS endpoint and ensure you can communicate with the LLMs hosted there.

With the connection working, let’s move on to see what models are available.

In [6]:
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}
data = {
    "model": "granite-3-2-8b-instruct",
    "messages": [{"role": "user", "content": "Hello, world!"}]
}

response = requests.post(url_chat, headers=headers, json=data)
print(response.json())

{'id': 'chatcmpl-21c5b0876e6d47f2981659e5c7c72fec', 'created': 1771961763, 'model': 'granite-3-2-8b-instruct', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'Hello! How can I assist you today?', 'role': 'assistant', 'tool_calls': None, 'function_call': None}, 'provider_specific_fields': {'stop_reason': None}}], 'usage': {'completion_tokens': 10, 'prompt_tokens': 12, 'total_tokens': 22, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'service_tier': None, 'prompt_logprobs': None, 'kv_transfer_params': None}


## 2.3 Listing the Models Available to the API Key

In [7]:
headers = {"Authorization": f"Bearer {key}"}

response = requests.get(url_models, headers=headers)
models = response.json()

print("Models available")

# Loop through and print just the names
for model in models['data']:
    print(f"\t- {model['id']}")

Models available
	- granite-4-0-h-tiny
	- qwen3-14b
	- microsoft-phi-4
	- granite-3-2-8b-instruct


## 2.4 Testing the Models’ Existing Knowledge


Ask a common RPG question


Next, we will test the language models’ innate knowledge of our name space. This is what many customers would do. Run the following code in a new cell to define the `test_all_models` function.


Now, run the following code to see what the models know about the Basic Fantasy game


In [8]:
def test_all_models(api_key: str, base_url: str, question: str, max_tokens: int = 1000):

    url_models = f"{base_url}/models"
    url_chat = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # 1. Get all models
    try:
        response = requests.get(url_models, headers=headers)
        response.raise_for_status()
        models_data = response.json().get("data", [])
    except Exception as e:
        print(f"❌ Failed to fetch models: {e}")
        return

    print(f"\n--- Running prompt against {len(models_data)} models ---\n")

    # 2. Loop through each model
    for model in models_data:
        model_id = model["id"]
        print(f"🤖 Testing Model: {model_id}...")

        payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": question}],
            "max_tokens": max_tokens
        }

        try:
            res = requests.post(url_chat, headers=headers, json=payload)
            res.raise_for_status()

            answer = res.json()["choices"][0]["message"]["content"]
            print(f"✅ Response:\n{answer}\n")

        except Exception as e:
            print(f"❌ Failed to get response from {model_id}: {e}\n")

    print("--- All tests complete ---")

Now, run the following code to see what the models know about the Basic Fantasy game


In [9]:
test_all_models(
    api_key=key,
    base_url=endpoint_base,
    question="In Basic Fantasy What happens if a Thief fails an Open Locks attempt?"
)


--- Running prompt against 4 models ---

🤖 Testing Model: granite-4-0-h-tiny...
✅ Response:
In Basic Fantasy Role-Playing Game, if a Thief fails an Open Locks attempt, several outcomes may occur depending on the narrative and system design used. However, generally, there are a few common responses:

1. The Lock Remains Secure: The lock simply does not open, and the thief has to find another way or devise another plan. This could include using different tools or techniques, such as a successful Disarm attempt or a successful Pick Lock attempt.

2. Failure Leaves Evidence: The thief might accidentally damage the lock in the process, making it weaker or requiring more time to unlock in the future. This could make the lock an utter failure.

3. Alertness Raised: Each failed attempt could raise the suspicion of any nearby guards or creatures, increasing the difficulty for subsequent attempts to open the same lock.

4. Finesse: Some game systems might also include a 'finesse' system, where 

You’ll notice that the output will vary slightly across each model, but the LLM will hedge their bets by saying something to the effect of “In many fantasy role-playing games, including Dungeons & Dragons,”  The customer’s game is not Dungeons & Dragons and we want specific to this game.

Given that the models’ innate answer mentions the customer’s top competitor is certainly a sore point. 

Once again, the models offer results that generalize based on whatever public documents it had access to during training. 

For reference, on page 9 of the rule book, the correct answer is below: 

>Open Locks allows the Thief to unlock a lock without a proper key. It may only be tried once per lock. If the attempt fails, the Thief must wait until they have gained another level of experience before trying again.`

Clearly, we need to augment the model’s response with data specific to the customer’s data and documents. In order to do that, we need to ingest the documents and make them ready for an LLM to consume.
